# Chạy thử UTMOSv2 trên tiếng Việt

**Mục tiêu:** chạy mã nguồn [UTMOSv2](https://github.com/sarulab-speech/UTMOSv2) (thầy gửi) và kiểm tra xem dự đoán MOS (độ tự nhiên của giọng nói) có kém chính xác với tiếng Việt hay không — vì theo `docs/datasets.md` của repo, model được huấn luyện trên BVCC, Sarulab data, Blizzard Challenges và SOMOS, không có dữ liệu tiếng Việt.

**Notebook làm 5 việc, chạy từ trên xuống (Runtime → Run all):**

| Phần | Thí nghiệm | Câu hỏi |
|---|---|---|
| A | Giọng **người thật** Việt vs Anh (FLEURS, cùng câu dịch song song) | Giọng người thật tiếng Việt có bị chấm thấp hơn một cách hệ thống không? |
| B | **Cùng hệ TTS** đọc cùng câu bằng 2 thứ tiếng (Edge neural, MMS-TTS, gTTS) | Thứ hạng chất lượng các hệ TTS có giữ được trên tiếng Việt không? Tiếng Việt có bị chấm lệch không? |
| C | Thêm **nhiễu** có kiểm soát (SNR 20/10/5/0 dB) | Model có còn phản ứng đúng chiều với suy giảm chất lượng trên tiếng Việt không? |
| D | **Khảo sát người nghe** trên web: 20 clip tiếng Việt, chấm độ tự nhiên 1–5 (ACR) | Điểm model tương quan với người nghe Việt đến đâu? → bằng chứng trực tiếp |
| E | Sinh **báo cáo** (`report.md` + biểu đồ + zip) | Gửi thầy |

A–C là bằng chứng gián tiếp (không cần người chấm). **D là bằng chứng trực tiếp** — cần nhóm nghe và chấm điểm, sau đó chạy lại phần D–E.

> Cần GPU: **Runtime → Change runtime type → T4 GPU**. Thời gian chạy A–C khoảng 15–25 phút (phần lớn là tải FLEURS và sinh TTS).

In [ ]:
# 0. Cài đặt
!nvidia-smi -L || echo "⚠️ Không thấy GPU — vào Runtime > Change runtime type > T4 GPU (CPU vẫn chạy được nhưng rất chậm)"
!pip -q install git+https://github.com/sarulab-speech/UTMOSv2.git edge-tts gTTS soundfile tabulate
!git clone -q https://github.com/sarulab-speech/UTMOSv2.git /content/UTMOSv2_src 2>/dev/null || true
!cd /content/UTMOSv2_src && git rev-parse --short HEAD > /content/utmosv2_commit.txt && cat /content/utmosv2_commit.txt

In [ ]:
# 1. Cấu hình
from pathlib import Path
import os, random, subprocess, tarfile, csv, json, shutil, warnings
import numpy as np, pandas as pd, soundfile as sf, librosa
warnings.filterwarnings("ignore")

N_SENT = 30                 # số câu song song Việt–Anh dùng cho A–C
SNR_LEVELS = [20, 10, 5, 0] # dB
PREDICT_DATASET = "sarulab" # mặc định của UTMOSv2; xem ghi chú ở phần báo cáo
SEED = 42
SR = 16000

# Link repo GitHub chứa web khảo sát (để chấm đúng 20 clip đang deploy), ví dụ "https://github.com/ten/repo"
SURVEY_REPO = ""

# Lưu mọi thứ vào Google Drive: phiên Colab ngắt vẫn còn, chạy lại sẽ bỏ qua bước đã xong
try:
    from google.colab import drive
    drive.mount("/content/drive"); WORK = Path("/content/drive/MyDrive/utmos_vi")
except Exception:
    WORK = Path("/content/utmos_vi")
AUDIO = WORK / "audio"; OUT = WORK / "outputs"
print("Thư mục làm việc:", WORK)
for p in (WORK, AUDIO, OUT): p.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

FLEURS = {"vi": "vi_vn", "en": "en_us"}
EDGE_VOICE = {"vi": "vi-VN-HoaiMyNeural", "en": "en-US-AriaNeural"}
MMS_MODEL = {"vi": "facebook/mms-tts-vie", "en": "facebook/mms-tts-eng"}

## Dữ liệu: FLEURS (giọng người thật, song song đa ngữ)
FLEURS là bộ đọc **cùng một tập câu** (dịch từ FLoRes) bằng ~100 ngôn ngữ, ghi âm theo cùng quy trình → so Việt–Anh trên cùng nội dung, cùng điều kiện thu, giảm nhiễu do chất lượng thu âm khác nhau.

In [ ]:
# 2. Tải FLEURS test split (vi_vn, en_us) và chọn N_SENT câu chung
BASE = "https://huggingface.co/datasets/google/fleurs/resolve/main/data"
COLS = ["id", "file_name", "raw", "norm", "chars", "num_samples", "gender"]
raw_dir = Path("/content/fleurs_raw"); raw_dir.mkdir(exist_ok=True)   # file tải tạm, không lưu Drive

def wget(url, out):
    if not out.exists():
        subprocess.run(["wget", "-q", "-O", str(out), url], check=True)

def n_wav(d): return len(list(Path(d).glob("*.wav"))) if Path(d).exists() else 0
SJ = OUT / "sentences.json"
if SJ.exists() and all(n_wav(AUDIO / l / "natural") >= 1 for l in FLEURS):
    sentences = {l: {int(k): v for k, v in d.items()} for l, d in json.load(open(SJ)).items()}
    print("Đã có dữ liệu FLEURS trên Drive → bỏ qua tải lại")
else:
  meta = {}
  for lang, code_ in FLEURS.items():
      tsv, tgz = raw_dir / f"{lang}_test.tsv", raw_dir / f"{lang}_test.tar.gz"
      wget(f"{BASE}/{code_}/test.tsv", tsv)
      wget(f"{BASE}/{code_}/audio/test.tar.gz", tgz)
      df = pd.read_csv(tsv, sep="\t", header=None, names=COLS, usecols=range(7),
                       quoting=csv.QUOTE_NONE, on_bad_lines="skip")
      df["dur"] = df["num_samples"] / SR
      meta[lang] = df[(df.dur >= 3) & (df.dur <= 15)].drop_duplicates("id")

  common = sorted(set(meta["vi"].id) & set(meta["en"].id))
  sent_ids = sorted(random.sample(common, min(N_SENT, len(common))))
  print(f"Câu chung Việt–Anh: {len(common)} → chọn {len(sent_ids)}")

  sentences = {}
  for lang in FLEURS:
      sel = meta[lang].set_index("id").loc[sent_ids]
      wanted = {r.file_name: i for i, r in sel.iterrows()}
      sentences[lang] = sel["raw"].to_dict()
      out_dir = AUDIO / lang / "natural"; out_dir.mkdir(parents=True, exist_ok=True)
      with tarfile.open(raw_dir / f"{lang}_test.tar.gz", "r:gz") as tar:
          for m in tar:
              name = os.path.basename(m.name)
              if name in wanted:
                  y, _ = librosa.load(tar.extractfile(m), sr=SR)
                  sf.write(out_dir / f"{wanted.pop(name)}.wav", y, SR)
              if not wanted: break
      print(lang, "giọng người thật:", len(list(out_dir.glob("*.wav"))), "file", "| thiếu:", len(wanted))

  json.dump(sentences, open(OUT / "sentences.json", "w"), ensure_ascii=False, indent=1)

## Sinh TTS: cùng hệ, cùng câu, hai thứ tiếng
Ba hệ có chất lượng khác nhau rõ: **Edge neural** (Microsoft, cao) · **MMS-TTS** (Meta, VITS đa ngữ, trung bình) · **gTTS** (Google Translate TTS, thấp hơn). Nếu model hoạt động tốt, thứ hạng nên giống nhau ở cả hai thứ tiếng.

In [ ]:
# 3. TTS
import torch
from gtts import gTTS
from transformers import VitsModel, AutoTokenizer
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def to_wav(src, dst):
    y, _ = librosa.load(src, sr=SR); sf.write(dst, y, SR)

def run_edge(lang):
    d = AUDIO / lang / "edge"; d.mkdir(exist_ok=True)
    for i, t in sentences[lang].items():
        mp3 = d / f"{i}.mp3"
        subprocess.run(["edge-tts", "--voice", EDGE_VOICE[lang], "--text", t, "--write-media", str(mp3)],
                       check=True, capture_output=True)
        to_wav(mp3, d / f"{i}.wav"); mp3.unlink()

def run_gtts(lang):
    d = AUDIO / lang / "gtts"; d.mkdir(exist_ok=True)
    for i, t in sentences[lang].items():
        mp3 = d / f"{i}.mp3"; gTTS(t, lang=lang).save(str(mp3))
        to_wav(mp3, d / f"{i}.wav"); mp3.unlink()

def run_mms(lang):
    d = AUDIO / lang / "mms"; d.mkdir(exist_ok=True)
    tok = AutoTokenizer.from_pretrained(MMS_MODEL[lang])
    if getattr(tok, "is_uroman", False):
        raise RuntimeError("model cần uroman — bỏ qua")
    m = VitsModel.from_pretrained(MMS_MODEL[lang]).to(DEV).eval()
    torch.manual_seed(SEED)
    for i, t in sentences[lang].items():
        with torch.no_grad():
            w = m(**tok(t, return_tensors="pt").to(DEV)).waveform[0].cpu().numpy()
        sf.write(d / f"{i}.wav", librosa.resample(w, orig_sr=m.config.sampling_rate, target_sr=SR), SR)
    del m; torch.cuda.empty_cache()

for lang in FLEURS:
    for name, fn in [("edge", run_edge), ("mms", run_mms), ("gtts", run_gtts)]:
        if n_wav(AUDIO / lang / name) >= len(sentences[lang]):
            print(f"✓ {lang}/{name} (đã có)"); continue
        try:
            fn(lang); print(f"✓ {lang}/{name}")
        except Exception as e:
            print(f"✗ {lang}/{name}: {e}")
            shutil.rmtree(AUDIO / lang / name, ignore_errors=True)

In [ ]:
# 4. Suy giảm có kiểm soát: thêm nhiễu trắng vào giọng người thật
rng = np.random.default_rng(SEED)
for lang in FLEURS:
    for snr in SNR_LEVELS:
        d = AUDIO / lang / f"noise{snr}"; d.mkdir(exist_ok=True)
        if n_wav(d) >= n_wav(AUDIO / lang / "natural"): continue
        for f in sorted((AUDIO / lang / "natural").glob("*.wav")):
            y, _ = sf.read(f)
            p = np.mean(y ** 2) / (10 ** (snr / 10))
            z = y + rng.normal(0, np.sqrt(p), len(y))
            sf.write(d / f.name, z / max(1.0, np.abs(z).max()), SR)
print({lang: sorted(p.name for p in (AUDIO / lang).iterdir()) for lang in FLEURS})

## Chạy UTMOSv2
Dùng đúng API trong README: `utmosv2.create_model(pretrained=True)` → `model.predict(input_dir=...)`. Cấu hình mặc định `fusion_stage3`, fold 0.

In [ ]:
# 5. Dự đoán MOS cho mọi thư mục
import utmosv2
model = utmosv2.create_model(pretrained=True)

def score_dir(d):
    return model.predict(input_dir=str(d), predict_dataset=PREDICT_DATASET,
                         device=DEV, num_workers=2, batch_size=16, verbose=False)

rows = []
for lang in ([] if (OUT / "scores.csv").exists() else FLEURS):
    for d in sorted((AUDIO / lang).iterdir()):
        if not any(d.glob("*.wav")): continue
        res = score_dir(d)
        for r in res:
            rows.append({"lang": lang, "system": d.name, "sent_id": Path(r["file_path"]).stem,
                         "pred_mos": float(r["predicted_mos"])})
        print(f"{lang}/{d.name}: {len(res)} file")

if rows: pd.DataFrame(rows).to_csv(OUT / "scores.csv", index=False)
else: print("Đã có scores.csv trên Drive → bỏ qua chấm lại")
scores = pd.read_csv(OUT / "scores.csv", dtype={"sent_id": str})
scores.groupby(["system", "lang"]).pred_mos.mean().unstack().round(2)

In [ ]:
# 6. Demo nhanh: nghe và xem điểm vài mẫu (cùng một câu)
from IPython.display import Audio, display, Markdown
sid = scores.sent_id.iloc[0]
for lang in FLEURS:
    display(Markdown(f"**{lang}** — “{sentences[lang][int(sid)]}”"))
    for system in ["natural", "edge", "mms", "gtts", "noise5"]:
        f = AUDIO / lang / system / f"{sid}.wav"
        if f.exists():
            s = scores.query("lang==@lang and system==@system and sent_id==@sid").pred_mos.iloc[0]
            print(f"  {system:8s} pred MOS = {s:.2f}"); display(Audio(str(f)))

In [ ]:
# 6b. (Tuỳ chọn) chấm file của bạn: upload .wav bất kỳ
# from google.colab import files
# for name in files.upload(): print(name, model.predict(input_path=name, predict_dataset=PREDICT_DATASET))

## Phân tích A–C
- **A/B:** so Việt vs Anh theo **cặp cùng câu** (Wilcoxon signed-rank), kèm chênh lệch trung bình và tỉ lệ cặp mà tiếng Việt bị chấm thấp hơn.
- **B:** thứ hạng các hệ TTS theo từng ngôn ngữ.
- **C:** tương quan Spearman giữa mức nhiễu và điểm (model tốt → càng nhiễu càng thấp, ρ gần 1 theo SNR).

In [ ]:
# 7. Phân tích A–C  (CELL:analysis)
from scipy import stats
import matplotlib.pyplot as plt

scores = pd.read_csv(OUT / "scores.csv", dtype={"sent_id": str})
ORDER = [s for s in ["natural", "edge", "mms", "gtts"] + [f"noise{x}" for x in SNR_LEVELS]
         if s in set(scores.system)]

summary = (scores.groupby(["system", "lang"]).pred_mos.agg(["mean", "std", "count"])
           .unstack("lang").reindex(ORDER))
summary.columns = [f"{lang}_{stat}" for stat, lang in summary.columns]
summary = summary[[c for c in ["vi_mean", "en_mean", "vi_std", "en_std", "vi_count"] if c in summary]]

paired = []
for s in ORDER:
    w = scores[scores.system == s].pivot_table(index="sent_id", columns="lang", values="pred_mos").dropna()
    if len(w) < 5 or not {"vi", "en"} <= set(w.columns): continue
    diff = w["vi"] - w["en"]
    p = stats.wilcoxon(w["vi"], w["en"]).pvalue if (diff != 0).any() else 1.0
    paired.append({"system": s, "n_pairs": len(w), "mean_vi": w["vi"].mean(), "mean_en": w["en"].mean(),
                   "diff_vi_minus_en": diff.mean(), "pct_vi_lower": (diff < 0).mean() * 100, "wilcoxon_p": p})
paired = pd.DataFrame(paired)

tts = [s for s in ["natural", "edge", "mms", "gtts"] if s in ORDER]
rank = {lang: scores[(scores.lang == lang) & scores.system.isin(tts)].groupby("system").pred_mos.mean()
        .sort_values(ascending=False).index.tolist() for lang in ["vi", "en"]}

snr_x = {"natural": 99, **{f"noise{x}": x for x in SNR_LEVELS}}
deg = []
for lang in ["vi", "en"]:
    d = scores[(scores.lang == lang) & scores.system.isin(snr_x)].copy()
    if d.empty: continue
    d["snr"] = d.system.map(snr_x)
    rho, p = stats.spearmanr(d.snr, d.pred_mos)
    m = d.groupby("snr").pred_mos.mean()
    deg.append({"lang": lang, "spearman_rho": rho, "p": p, "clean": m.get(99, np.nan),
                "snr0": m.get(0, np.nan), "drop_clean_to_0dB": m.get(99, np.nan) - m.get(0, np.nan)})
deg = pd.DataFrame(deg)

# Biểu đồ
fig, ax = plt.subplots(figsize=(9, 4))
pos = np.arange(len(ORDER))
for k, (lang, c) in enumerate([("vi", "#c0392b"), ("en", "#2c6fbb")]):
    data = [scores[(scores.system == s) & (scores.lang == lang)].pred_mos.values for s in ORDER]
    b = ax.boxplot(data, positions=pos + (k - 0.5) * 0.35, widths=0.3, patch_artist=True, showfliers=False)
    for patch in b["boxes"]: patch.set(facecolor=c, alpha=0.55)
    ax.plot([], [], color=c, lw=8, alpha=0.55, label={"vi": "Tiếng Việt", "en": "Tiếng Anh"}[lang])
ax.set_xticks(pos); ax.set_xticklabels(ORDER, rotation=20); ax.set_ylabel("MOS dự đoán (UTMOSv2)")
ax.set_title("Điểm UTMOSv2 theo hệ / điều kiện"); ax.legend(); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "fig_systems.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
for lang, c in [("vi", "#c0392b"), ("en", "#2c6fbb")]:
    d = scores[(scores.lang == lang) & scores.system.isin(snr_x)].copy(); d["snr"] = d.system.map(snr_x)
    m = d.groupby("snr").pred_mos.agg(["mean", "std"]).sort_index(ascending=False)
    ax.errorbar(range(len(m)), m["mean"], yerr=m["std"], marker="o", capsize=3, color=c,
                label={"vi": "Tiếng Việt", "en": "Tiếng Anh"}[lang])
ax.set_xticks(range(len(m))); ax.set_xticklabels(["sạch" if s == 99 else f"{s} dB" for s in m.index])
ax.set_xlabel("Mức nhiễu (SNR)"); ax.set_ylabel("MOS dự đoán"); ax.set_title("Phản ứng với nhiễu")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(OUT / "fig_noise.png", dpi=150); plt.show()

display(summary.round(2)); display(paired.round(3)); print("Thứ hạng:", rank); display(deg.round(3))

## D. Khảo sát người nghe (bằng chứng trực tiếp)
Cell 8 tạo bảng nối `key.csv` (clip → hệ, câu, điểm UTMOSv2), lưu trên Drive và **không gửi cho người chấm**.
- **Đã deploy web rồi:** điền `SURVEY_REPO` ở cell 1. Cell 8 tải 20 clip từ repo, cho UTMOSv2 chấm trực tiếp đúng file người nghe nghe, và dò lại mỗi clip thuộc hệ nào.
- **Chưa deploy:** để trống `SURVEY_REPO`. Cell 8 chọn 20 mẫu có điểm trải đều từ thấp đến cao và xuất `survey_clips.zip` (MP3).

Giải nén `survey_clips.zip` vào thư mục `clips/` của web khảo sát. Mỗi người chấm độ tự nhiên 1–5 cho từng clip (thang ACR), xong tải về `ratings_<tên>.csv`; upload **cả 4 file cùng lúc** ở cell 9.

In [ ]:
# 8. Bộ 20 clip khảo sát  (CELL:export)
#  - Đã có key.csv trên Drive  → dùng lại, không làm gì.
#  - Có SURVEY_REPO            → chấm UTMOSv2 trực tiếp 20 clip đang deploy + dò lại clip thuộc hệ nào.
#  - Chưa có gì                → chọn 20 mẫu mới và xuất survey_clips.zip để deploy.
from scipy import signal
N_SURVEY = 20
KEY = OUT / "key.csv"
en = scores[scores.lang == "en"].set_index(["system", "sent_id"]).pred_mos
def vi_minus_en(system, sid): return round(float(scores[(scores.lang == "vi") & (scores.system == system) & (scores.sent_id == sid)].pred_mos.iloc[0]) - en.get((system, sid), np.nan), 3)

def match_snr(y, c, max_lag=2000):
    # Độ giống giữa clip deploy (y) và một file gốc (c): SNR (dB) sau khi căn lệch và chỉnh âm lượng
    corr = signal.correlate(y, c, mode="full", method="fft"); lags = signal.correlation_lags(len(y), len(c))
    m = np.abs(lags) <= max_lag; lag = int(lags[m][np.argmax(corr[m])])
    a_, b_ = (y[lag:], c) if lag >= 0 else (y, c[-lag:])
    n = min(len(a_), len(b_)); a_, b_ = a_[:n], b_[:n]
    g = (a_ @ b_) / (b_ @ b_ + 1e-12); res = a_ - g * b_
    return 10 * np.log10((a_ @ a_) / (res @ res + 1e-12))

if KEY.exists():
    key = pd.read_csv(KEY, dtype={"sent_id": str}); print("Dùng key.csv đã lưu trên Drive")
elif SURVEY_REPO:
    rep = Path("/content/survey_repo"); shutil.rmtree(rep, ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", SURVEY_REPO, str(rep)], check=True)
    tmp = Path("/content/survey_wav"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
    for f in sorted((rep / "clips").glob("clip_*.mp3")):
        y, _ = librosa.load(f, sr=SR); sf.write(tmp / f"{f.stem}.wav", y, SR)
    pred = {Path(r["file_path"]).stem: float(r["predicted_mos"]) for r in score_dir(tmp)}
    cands = [(d.name, f.stem, sf.read(f)[0]) for d in sorted((AUDIO / "vi").iterdir()) for f in sorted(d.glob("*.wav"))]
    rows = []
    for f in sorted(tmp.glob("*.wav")):
        y, _ = sf.read(f)
        ranked = sorted(((match_snr(y, c), s_, i) for s_, i, c in cands if abs(len(c) - len(y)) < SR // 2), reverse=True)
        best = ranked[0] if ranked else (np.nan, "?", "?")
        margin = ranked[0][0] - ranked[1][0] if len(ranked) > 1 else np.nan
        rows.append({"clip_id": f.stem, "system": best[1], "sent_id": best[2], "pred_mos": pred[f.stem],
                     "match_margin_db": round(margin, 1),
                     "vi_minus_en": vi_minus_en(best[1], best[2]) if best[1] != "?" else np.nan,
                     "text": sentences["vi"].get(int(best[2])) if best[2] != "?" else ""})
    key = pd.DataFrame(rows); key.to_csv(KEY, index=False)
    print("pred_mos = UTMOSv2 chấm trực tiếp file mp3 người nghe sẽ nghe.")
    weak = key[~(key.match_margin_db >= 3)]
    print("Dò hệ:", len(key) - len(weak), "/", len(key), "clip chắc chắn (hơn ứng viên thứ hai ≥ 3 dB).")
    if len(weak): print("⚠️ Không chắc:", list(weak.clip_id), "— cột system có thể sai (thường do audio TTS sinh lại khác lần trước); pred_mos vẫn đúng.")
else:
    vi = scores[scores.lang == "vi"].copy()
    vi["bin"] = pd.qcut(vi.pred_mos.rank(method="first"), N_SURVEY, labels=False)
    used_sys, used_sent, pick = {}, set(), []
    for b_, g in vi.groupby("bin"):
        g = g.sample(frac=1, random_state=SEED + int(b_))
        g = g.assign(_k=[(r.sent_id in used_sent, used_sys.get(r.system, 0)) for r in g.itertuples()]).sort_values("_k", kind="stable")
        r = g.iloc[0]; pick.append(r); used_sent.add(r.sent_id); used_sys[r.system] = used_sys.get(r.system, 0) + 1
    key = pd.DataFrame(pick).drop(columns=["_k", "bin", "lang"]).sample(frac=1, random_state=SEED + 1).reset_index(drop=True)
    key.insert(0, "clip_id", [f"clip_{k:02d}" for k in range(1, len(key) + 1)])
    key["vi_minus_en"] = [vi_minus_en(r.system, r.sent_id) for r in key.itertuples()]
    key["text"] = [sentences["vi"][int(i)] for i in key.sent_id]
    SV = WORK / "survey_clips"; shutil.rmtree(SV, ignore_errors=True); SV.mkdir()
    for r in key.itertuples():
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(AUDIO / "vi" / r.system / f"{r.sent_id}.wav"),
                        "-ac", "1", "-b:a", "128k", str(SV / f"{r.clip_id}.mp3")], check=True)
    key.to_csv(KEY, index=False)   # KHÔNG gửi cho người chấm
    shutil.make_archive(str(WORK / "survey_clips"), "zip", SV)
    try:
        from google.colab import files; files.download(str(WORK / "survey_clips.zip"))
    except Exception: pass

display(key.drop(columns=["text"], errors="ignore").sort_values("pred_mos").round(2))
print("Hệ:", key.system.value_counts().to_dict(), "| khoảng điểm:", round(key.pred_mos.min(), 2), "→", round(key.pred_mos.max(), 2))

In [ ]:
# 9. Sau khi khảo sát xong: upload các file ratings_*.csv rồi chạy cell này  (CELL:ratings)
CRIT = {"phat_am": "Phát âm", "thanh_dieu": "Thanh điệu", "nhip_ngat": "Nhịp & ngắt nghỉ",
        "ngu_dieu": "Ngữ điệu", "chat_luong": "Chất lượng âm thanh"}
# Upload cùng lúc cả 4 file ratings_<tên>.csv mà mỗi người tải về từ web
DETAIL = OUT / "ratings_detail.csv"
if not DETAIL.exists():
    try:
        from google.colab import files
        up = files.upload()
        if up:
            parts = [pd.read_csv(f) for f in up]
            pd.concat(parts, ignore_index=True).drop_duplicates(["rater", "clip_id"], keep="last").to_csv(DETAIL, index=False)
            print("Đã gộp", len(parts), "file")
    except Exception as e:
        print("Chưa có kết quả khảo sát:", e)

lt = crit_tbl = None
if DETAIL.exists():
    d = pd.read_csv(DETAIL)
    k = pd.read_csv(OUT / "key.csv", dtype={"sent_id": str})
    wide = d.pivot_table(index="clip_id", columns="rater", values="overall")
    raters = list(wide.columns)
    per_clip = d.groupby("clip_id")[["overall", *[c for c in CRIT if c in d]]].mean().rename(columns={"overall": "true_mos"})
    lt = per_clip.reset_index().merge(k, on="clip_id")
    lt.to_csv(OUT / "listening_merged.csv", index=False)
    pairs = [stats.spearmanr(wide[a], wide[b], nan_policy="omit")[0] for i, a in enumerate(raters) for b in raters[i + 1:]]
    sys_m = lt.groupby("system")[["true_mos", "pred_mos"]].mean()
    metrics = {
        "n_clips": len(lt), "n_raters": len(raters),
        "inter_rater_srcc_mean": float(np.mean(pairs)) if pairs else np.nan,
        "utt_LCC": stats.pearsonr(lt.true_mos, lt.pred_mos)[0],
        "utt_SRCC": stats.spearmanr(lt.true_mos, lt.pred_mos)[0],
        "utt_KTAU": stats.kendalltau(lt.true_mos, lt.pred_mos)[0],
        "utt_MSE": float(((lt.true_mos - lt.pred_mos) ** 2).mean()),
        "sys_SRCC": stats.spearmanr(sys_m.true_mos, sys_m.pred_mos)[0] if len(sys_m) > 2 else np.nan,
        "mean_bias_pred_minus_true": float((lt.pred_mos - lt.true_mos).mean()),
    }
    json.dump(metrics, open(OUT / "listening_metrics.json", "w"), indent=1)
    # Tiêu chí nào người nghe chấm thấp mà model không thấy?
    crit_tbl = pd.DataFrame([{"tiêu chí": v, "điểm TB (người)": lt[c].mean(),
                              "SRCC với MOS tổng (người)": stats.spearmanr(lt[c], lt.true_mos)[0],
                              "SRCC với UTMOSv2": stats.spearmanr(lt[c], lt.pred_mos)[0]}
                             for c, v in CRIT.items() if c in lt])
    crit_tbl = crit_tbl.set_index("tiêu chí") if len(crit_tbl) else None
    fig, ax = plt.subplots(figsize=(5, 5))
    for s, g in lt.groupby("system"): ax.scatter(g.true_mos, g.pred_mos, label=s, alpha=.8)
    ax.plot([1, 5], [1, 5], "k--", lw=1); ax.set_xlim(1, 5); ax.set_ylim(1, 5)
    ax.set_xlabel("MOS người nghe Việt"); ax.set_ylabel("MOS dự đoán (UTMOSv2)")
    ax.set_title(f"SRCC={metrics['utt_SRCC']:.2f} · LCC={metrics['utt_LCC']:.2f}"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUT / "fig_listening.png", dpi=150); plt.show()
    display(sys_m.round(2)); print(json.dumps(metrics, indent=1))

## E. Sinh báo cáo
Báo cáo chỉ ghi **số liệu đo được**. Phần *Nhận xét* để nhóm điền sau khi xem kết quả.

In [ ]:
# 10. Sinh report.md + zip kết quả  (CELL:report)
def t(df): return df.to_markdown(floatfmt=".3f") if len(df) else "_(không có dữ liệu)_"
commit = (Path("/content/utmosv2_commit.txt").read_text().strip()
          if Path("/content/utmosv2_commit.txt").exists() else "?")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
L = ["# Báo cáo chạy thử UTMOSv2 trên tiếng Việt", "",
     "## 1. Thiết lập",
     f"- Mã nguồn: github.com/sarulab-speech/UTMOSv2 @ `{commit}`; model `fusion_stage3`, fold 0, "
     f"`predict_dataset='{PREDICT_DATASET}'`; phần cứng: {gpu}.",
     "- Dữ liệu huấn luyện của UTMOSv2 (theo `docs/datasets.md`): BVCC, Sarulab data, Blizzard Challenges, SOMOS — không có tiếng Việt.",
     f"- Dữ liệu thử: {scores.sent_id.nunique()} câu song song Việt–Anh từ FLEURS (test); "
     "giọng người thật + 3 hệ TTS (Edge neural, MMS-TTS, gTTS) + nhiễu trắng SNR "
     + "/".join(map(str, SNR_LEVELS)) + " dB.", "",
     "## 2. Điểm dự đoán theo hệ và ngôn ngữ", t(summary.round(3)), "", "![](fig_systems.png)", "",
     "## 3. So sánh cặp cùng câu (Việt − Anh, Wilcoxon signed-rank)", t(paired.set_index("system")), "",
     "## 4. Thứ hạng hệ TTS theo điểm dự đoán",
     f"- Tiếng Việt: {' > '.join(rank.get('vi', []))}", f"- Tiếng Anh: {' > '.join(rank.get('en', []))}", "",
     "## 5. Phản ứng với nhiễu", t(deg.set_index("lang")), "", "![](fig_noise.png)", ""]
if lt is not None:
    L += ["## 6. Đối chiếu với người nghe Việt (ACR 1–5)",
          t(pd.Series(json.load(open(OUT / "listening_metrics.json"))).to_frame("giá trị")), "",
          t(lt.groupby("system")[["true_mos", "pred_mos"]].mean()), "", "![](fig_listening.png)", ""]
else:
    L += ["## 6. Đối chiếu với người nghe Việt", "_Chưa có — chạy cell 9 sau khi khảo sát xong._", ""]
L += ["## 7. Nhận xét", "_(Nhóm điền sau khi xem kết quả.)_", "",
      "## 8. Hạn chế",
      "- Phần 2–5 là bằng chứng gián tiếp: chênh lệch Việt–Anh có thể đến từ người đọc/điều kiện thu, không chỉ từ ngôn ngữ.",
      "- Khảo sát 20 mẫu do thành viên nhóm chấm: ít người, không phải người nghe ngẫu nhiên; cần mở rộng nếu đưa vào khoá luận.",
      f"- Chỉ dùng một cấu hình (`fusion_stage3`, fold 0, `predict_dataset='{PREDICT_DATASET}'`)."]
(OUT / "report.md").write_text("\n".join(L), encoding="utf-8")
shutil.make_archive(str(WORK / "utmosv2_vi_results"), "zip", OUT)
print("\n".join(L[:12])); print("...\n→", WORK / "utmosv2_vi_results.zip")
try:
    from google.colab import files; files.download(str(WORK / "utmosv2_vi_results.zip"))
except Exception: pass